In [1]:
# =========================================================
# PS S6E4 - exp_20260410_041_trompt_strict_min
# strict Trompt minimum reproduction
#
# Key fixes vs source notebook:
# - feature engineering fit is moved inside each fold
# - scaler fit is moved inside each fold
# - KBinsDiscretizer fit is moved inside each fold
# - proper OOF / test probability saving
# - feature state is train-fold-only
# =========================================================

## Import / config

In [2]:
# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
!pip install -q pytorch_frame

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.3/146.3 kB 3.1 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import random
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, KBinsDiscretizer

import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler

import torch_frame
from torch_frame import stype
from torch_frame.data.loader import DataLoader
from torch_frame.nn.models.trompt import Trompt
from torch_frame.data.stats import compute_col_stats
from torch_frame.data import DataFrameToTensorFrameConverter

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260410_041_trompt_strict_min"

    SEED = 42
    FOLDS = 5

    EPOCHS = 6
    LR = 1e-3
    WD = 1e-4
    WARMUP = 0.15
    BATCH = 512

    CHANNELS = 64
    PROMPTS = 8
    LAYERS = 4

    ID_COL = "id"
    TARGET = "Irrigation_Need"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    # same minimal FE as source notebook
    CYCLIC_RAINFALL_PERIODS = [100]
    BIN_CONFIG = {
        "Temperature_C": [5, 40],
    }
    BIN_STRATEGIES = ["quantile"]

## Utils

In [4]:
# ---------------------------------------------------------
# Utilities
# ---------------------------------------------------------
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def get_feature_columns(df: pd.DataFrame):
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols]
    return cat_cols, num_cols


class FoldFeatureState:
    def __init__(self):
        self.category_map = {}
        self.kbin_map = {}
        self.scaler = None
        self.base_cat_cols = None
        self.base_num_cols = None
        self.new_cat_cols = None
        self.new_num_cols = None


def fit_transform_features(train_df: pd.DataFrame, test_like_df: pd.DataFrame):
    """
    Fit FE state on train_df only, then transform both train_df and test_like_df.
    Returns transformed dfs and fitted state.
    """
    state = FoldFeatureState()

    train_df = train_df.copy()
    test_like_df = test_like_df.copy()

    base_cat_cols, base_num_cols = get_feature_columns(train_df)
    state.base_cat_cols = base_cat_cols.copy()
    state.base_num_cols = base_num_cols.copy()

    # 1) cyclic encoding
    for p in CFG.CYCLIC_RAINFALL_PERIODS:
        col_name = f"_Rainfall_mm_sin_{p}"
        train_df[col_name] = np.sin(2 * np.pi * train_df["Rainfall_mm"] / p).astype("float32")
        test_like_df[col_name] = np.sin(2 * np.pi * test_like_df["Rainfall_mm"] / p).astype("float32")

    # 2) categorize numericals with floor, fit on train only
    for col in base_num_cols:
        cat_name = f"{col}_cat_"

        train_floor = np.floor(train_df[col])
        train_codes, train_uniques = pd.factorize(train_floor)
        state.category_map[col] = pd.Index(train_uniques)

        code_map = {cat: i for i, cat in enumerate(train_uniques)}

        test_floor = np.floor(test_like_df[col])
        test_codes = test_floor.map(code_map).fillna(-1).astype("int32")

        train_df[cat_name] = pd.Series(train_codes, index=train_df.index).astype("category")
        test_like_df[cat_name] = test_codes.astype("category")

    # 3) discretize selected numericals, fit on train only
    for col, bins_list in CFG.BIN_CONFIG.items():
        for n_bins in bins_list:
            for strategy in CFG.BIN_STRATEGIES:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"

                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode="ordinal",
                    strategy=strategy,
                    subsample=None,
                )
                train_binned = kb.fit_transform(train_df[[col]]).ravel().astype("int32")
                test_binned = kb.transform(test_like_df[[col]]).ravel().astype("int32")

                state.kbin_map[bin_name] = kb

                train_df[bin_name] = pd.Series(train_binned, index=train_df.index).astype("category")
                test_like_df[bin_name] = pd.Series(test_binned, index=test_like_df.index).astype("category")

    # 4) collect new cols
    new_cat_cols = [c for c in train_df.columns if c.endswith("_")]
    new_num_cols = [c for c in train_df.columns if c.startswith("_")]

    state.new_cat_cols = new_cat_cols.copy()
    state.new_num_cols = new_num_cols.copy()

    final_cat_cols = base_cat_cols + new_cat_cols
    final_num_cols = [c for c in base_num_cols + new_num_cols if c not in final_cat_cols]

    # 5) min-max scale numericals, fit on train only
    scaler = MinMaxScaler()
    train_df[final_num_cols] = scaler.fit_transform(train_df[final_num_cols]).astype(np.float32)
    test_like_df[final_num_cols] = scaler.transform(test_like_df[final_num_cols]).astype(np.float32)
    state.scaler = scaler

    return train_df, test_like_df, final_cat_cols, final_num_cols, state


def build_col_to_stype(cat_cols, num_cols, target_col):
    col_to_stype = {c: stype.numerical for c in num_cols}
    col_to_stype.update({c: stype.categorical for c in cat_cols})
    col_to_stype[target_col] = stype.numerical
    return col_to_stype


def predict_probs(model, loader, device):
    model.eval()
    preds_all = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch).mean(dim=1)
            probs = torch.softmax(out, dim=-1)
            preds_all.append(probs.cpu().numpy())

    return np.concatenate(preds_all, axis=0)

## Main

In [5]:
# ---------------------------------------------------------
# Main
# ---------------------------------------------------------
seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch       version:", torch.__version__)
print("PyTorch Frame version:", torch_frame.__version__)
print("Device:", device)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)

X_full = train.drop([CFG.ID_COL, CFG.TARGET], axis=1).copy()
y_full = train[CFG.TARGET].copy()
X_test_full = test.drop([CFG.ID_COL], axis=1).copy()
test_id = test[CFG.ID_COL].copy()

print("X_full shape:", X_full.shape)
print("X_test shape:", X_test_full.shape)

skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)

n_classes = y_full.nunique()
oof_preds = np.zeros((len(X_full), n_classes), dtype=np.float32)
test_preds = np.zeros((len(X_test_full), n_classes), dtype=np.float32)

fold_scores = []
feature_meta_rows = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full), 1):
    print(f"\n--- Fold {fold}/{CFG.FOLDS} ---")

    X_tr_raw = X_full.iloc[tr_idx].reset_index(drop=True)
    X_val_raw = X_full.iloc[val_idx].reset_index(drop=True)
    X_tst_raw = X_test_full.reset_index(drop=True)

    y_tr = y_full.iloc[tr_idx].reset_index(drop=True)
    y_val = y_full.iloc[val_idx].reset_index(drop=True)

    # strict FE: fit only on train fold
    X_tr, X_val, cat_cols, num_cols, state = fit_transform_features(X_tr_raw, X_val_raw)
    _, X_tst, _, _, _ = fit_transform_features(X_tr_raw, X_tst_raw)

    print(f"  len(cat_cols)={len(cat_cols)}, len(num_cols)={len(num_cols)}")

    # build fold dfs with target
    train_df = pd.concat([X_tr, pd.Series(y_tr, name=CFG.TARGET)], axis=1)
    val_df = pd.concat([X_val, pd.Series(y_val, name=CFG.TARGET)], axis=1)

    col_to_stype = build_col_to_stype(cat_cols, num_cols, CFG.TARGET)

    col_stats = {
        c: compute_col_stats(train_df[c], stype=col_to_stype[c])
        for c in train_df.columns
        if c in col_to_stype
    }

    converter = DataFrameToTensorFrameConverter(
        col_stats=col_stats,
        target_col=CFG.TARGET,
        col_to_stype=col_to_stype,
    )

    tf_train = converter(train_df)
    tf_val = converter(val_df)
    tf_test = converter(X_tst)

    class_counts = np.bincount(y_tr.values, minlength=n_classes)
    weights = 1.0 / class_counts
    sample_weights = weights[y_tr.values]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(tf_train, batch_size=CFG.BATCH, sampler=sampler)
    val_loader = DataLoader(tf_val, batch_size=CFG.BATCH, shuffle=False)
    test_loader = DataLoader(tf_test, batch_size=CFG.BATCH, shuffle=False)

    model = Trompt(
        channels=CFG.CHANNELS,
        out_channels=n_classes,
        num_prompts=CFG.PROMPTS,
        num_layers=CFG.LAYERS,
        col_stats=col_stats,
        col_names_dict=converter.col_names_dict,
    ).to(device)

    total_steps = len(train_loader) * CFG.EPOCHS
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.LR,
        total_steps=total_steps,
        pct_start=CFG.WARMUP,
        cycle_momentum=False,
    )

    best_val_score = -1.0
    best_val_probs = None
    best_test_probs = None
    best_epoch = -1

    for epoch in range(1, CFG.EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_samples = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            batch_size, num_layers, num_classes = out.size()

            pred = out.view(-1, num_classes)
            y_rep = batch.y.long().repeat_interleave(num_layers)

            loss = F.cross_entropy(pred, y_rep)
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * batch_size
            total_samples += batch_size

        avg_loss = total_loss / (total_samples + 1e-12)

        val_probs = predict_probs(model, val_loader, device)
        val_score = balanced_accuracy_score(y_val.values, np.argmax(val_probs, axis=1))

        print(f"    Epoch {epoch:02d} - train_loss: {avg_loss:.5f} val_score: {val_score:.5f}")

        if val_score > best_val_score:
            best_val_score = val_score
            best_val_probs = val_probs.copy()
            best_test_probs = predict_probs(model, test_loader, device)
            best_epoch = epoch

    oof_preds[val_idx] = best_val_probs
    test_preds += best_test_probs / CFG.FOLDS
    fold_scores.append(float(best_val_score))

    feature_meta_rows.append({
        "fold": fold,
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        "best_epoch": best_epoch,
        "best_val_score": float(best_val_score),
    })

    print(f"  Fold {fold} best_epoch={best_epoch} best_val_score={best_val_score:.6f}")

    del model, optimizer, scheduler
    del tf_train, tf_val, tf_test
    del train_loader, val_loader, test_loader
    del X_tr_raw, X_val_raw, X_tst_raw, X_tr, X_val, X_tst
    del train_df, val_df, col_stats, converter
    gc.collect()
    torch.cuda.empty_cache()

oof_score = balanced_accuracy_score(y_full.values, np.argmax(oof_preds, axis=1))
print(f"\nOverall OOF Score: {oof_score:.6f}")

PyTorch       version: 2.10.0+cu128
PyTorch Frame version: 0.3.0
Device: cuda
X_full shape: (630000, 19)
X_test shape: (270000, 19)

--- Fold 1/5 ---
  len(cat_cols)=21, len(num_cols)=12
    Epoch 01 - train_loss: 0.41826 val_score: 0.96399
    Epoch 02 - train_loss: 0.10474 val_score: 0.97200
    Epoch 03 - train_loss: 0.08257 val_score: 0.97335
    Epoch 04 - train_loss: 0.07502 val_score: 0.97341
    Epoch 05 - train_loss: 0.06967 val_score: 0.97523
    Epoch 06 - train_loss: 0.06616 val_score: 0.97539
  Fold 1 best_epoch=6 best_val_score=0.975390

--- Fold 2/5 ---
  len(cat_cols)=21, len(num_cols)=12
    Epoch 01 - train_loss: 0.37939 val_score: 0.96523
    Epoch 02 - train_loss: 0.11390 val_score: 0.96628
    Epoch 03 - train_loss: 0.08648 val_score: 0.97260
    Epoch 04 - train_loss: 0.07695 val_score: 0.97468
    Epoch 05 - train_loss: 0.07120 val_score: 0.97424
    Epoch 06 - train_loss: 0.06748 val_score: 0.97500
  Fold 2 best_epoch=6 best_val_score=0.975004

--- Fold 3/5 ---


## Save outputs

In [6]:
# ---------------------------------------------------------
# Save artifacts
# ---------------------------------------------------------
np.save(f"{CFG.OUTPUT_DIR}/oof_trompt_strict_min_proba.npy", oof_preds)
np.save(f"{CFG.OUTPUT_DIR}/pred_trompt_strict_min_proba.npy", test_preds)

prob_cols = ["prob_Low", "prob_Medium", "prob_High"]

oof_df = pd.DataFrame(oof_preds, columns=prob_cols)
oof_df.insert(0, CFG.ID_COL, train[CFG.ID_COL].values)
oof_df.to_csv(f"{CFG.OUTPUT_DIR}/oof_trompt_strict_min.csv", index=False)

test_df = pd.DataFrame(test_preds, columns=prob_cols)
test_df.insert(0, CFG.ID_COL, test_id.values)
test_df.to_csv(f"{CFG.OUTPUT_DIR}/test_trompt_strict_min.csv", index=False)

submission = pd.DataFrame({
    CFG.ID_COL: test_id.values,
    CFG.TARGET: pd.Series(np.argmax(test_preds, axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_trompt_strict_min.csv", index=False)

feature_meta_df = pd.DataFrame(feature_meta_rows)
feature_meta_df.to_csv(f"{CFG.OUTPUT_DIR}/fold_feature_meta.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "model_family": "trompt",
    "metric": "balanced_accuracy_score",
    "seed": CFG.SEED,
    "n_splits": CFG.FOLDS,
    "epochs": CFG.EPOCHS,
    "fold_scores": fold_scores,
    "oof_score": float(oof_score),
    "notes": {
        "strict_cv": True,
        "fit_before_split_removed": True,
        "feature_engineering_in_fold": True,
        "scaler_in_fold": True,
        "kbins_in_fold": True,
        "weighted_sampler": True,
        "source_notebook": "vld_trompt strictified",
    },
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("\nSaved files:")
for fn in [
    "oof_trompt_strict_min_proba.npy",
    "pred_trompt_strict_min_proba.npy",
    "oof_trompt_strict_min.csv",
    "test_trompt_strict_min.csv",
    "submission_trompt_strict_min.csv",
    "fold_feature_meta.csv",
    "cv_result.json",
]:
    print("-", f"{CFG.OUTPUT_DIR}/{fn}")


Saved files:
- /kaggle/working/exp_20260410_041_trompt_strict_min/oof_trompt_strict_min_proba.npy
- /kaggle/working/exp_20260410_041_trompt_strict_min/pred_trompt_strict_min_proba.npy
- /kaggle/working/exp_20260410_041_trompt_strict_min/oof_trompt_strict_min.csv
- /kaggle/working/exp_20260410_041_trompt_strict_min/test_trompt_strict_min.csv
- /kaggle/working/exp_20260410_041_trompt_strict_min/submission_trompt_strict_min.csv
- /kaggle/working/exp_20260410_041_trompt_strict_min/fold_feature_meta.csv
- /kaggle/working/exp_20260410_041_trompt_strict_min/cv_result.json
